In [ ]:
# This sciprt generate the tables of percentages used to do the correspondance analysis in R:
# - per site and per year
# - per site over years



In [2]:
import pandas as pd
import os
import pickle
import numpy as np

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import MONITORING_DATA, classes18, fig_class_order

In [6]:
mp_pred_combined = MONITORING_DATA / "PREDICTIONS_51slides_combined/"
p_csv  = MONITORING_DATA / "Tables_for_figures" 

p_save_table = MONITORING_DATA / "Tables_PollenCounts/"

In [7]:
df_long = pd.read_csv(p_csv/ "comptages_brutauto_all_perslide.csv")

df_long.head()


,site,year,slide,class,count,class_num,site_year
0,D1,2019,D1_2019_A,QuercusDeciduous,29.147301,0,D1_2019
1,D1,2020,D1_2020_A,QuercusDeciduous,25.037277,0,D1_2020
2,D1,2020,D1_2020_B,QuercusDeciduous,88.580480,0,D1_2020
3,D1,2021,D1_2021_A,QuercusDeciduous,78.337110,0,D1_2021
4,D2,2020,D2_2020_B,QuercusDeciduous,49.359799,0,D2_2020


In [5]:
df_long_2 = df_long.drop(columns="slide")
df_all = df_long_2.groupby(["class", "site", "year", "site_year", 'class_num']).mean().reset_index() 
df_all.head()

,class,site,year,site_year,class_num,count
0,Buxus,D1,2019,D1_2019,2,53.806745
1,Buxus,D1,2020,D1_2020,2,20.822339
2,Buxus,D1,2021,D1_2021,2,24.990120
3,Buxus,D2,2020,D2_2020,2,13.853471
4,Buxus,D2,2021,D2_2021,2,30.323475


In [6]:

df_siteyear_perc = df_all[df_all["class"].isin(fig_class_order)]

df_siteyear_perc.loc[:, "proportion"] = df_siteyear_perc["count"] * 100 / df_siteyear_perc.groupby(["site", "year"])["count"].transform("sum")

df_siteyear_perc

,class,site,year,site_year,class_num,count,proportion
0,Buxus,D1,2019,D1_2019,2,53.806745,6.734485
1,Buxus,D1,2020,D1_2020,2,20.822339,1.226623
2,Buxus,D1,2021,D1_2021,2,24.990120,1.016147
3,Buxus,D2,2020,D2_2020,2,13.853471,0.795045
4,Buxus,D2,2021,D2_2021,2,30.323475,0.712326
...,...,...,...,...,...,...,...
499,VitisS,W4,2019,W4_2019,11,82.988265,3.651658
500,VitisS,W4,2020,W4_2020,11,80.493538,3.775502
501,VitisS,W4,2021,W4_2021,11,83.066276,2.248256
502,VitisS,W4,2022,W4_2022,11,121.282832,2.812331


In [7]:

df_siteyear_perc = df_siteyear_perc.pivot(index="site_year", columns="class", values="proportion").reset_index()
df_siteyear_perc.columns.name = None
df_siteyear_perc.set_index('site_year', inplace=True)

df_siteyear_perc = df_siteyear_perc[fig_class_order]
df_siteyear_perc.reset_index()

df_siteyear_perc.to_csv(p_save_table / "percentages_persiteperyear_forAC_withOther.csv")

df_siteyear_perc

,QuercusDeciduous,QuercusIlex,Buxus,Phillyrea,Fraxinus,Olea,Cupressaceae,Pistacia,Poaceae,Plantago,VitisF,VitisS,Pinaceae,Other
site_year,,,,,,,,,,,,,,
D1_2019,3.648094,9.803933,6.734485,0.714618,0.982000,1.217762,12.221043,13.452162,3.736070,1.032800,3.044098,1.998502,8.565359,32.849074
D1_2020,3.346554,7.682511,1.226623,1.365324,2.067332,1.657528,12.321349,21.556190,6.342098,1.415773,1.621260,0.533793,7.274859,31.588805
D1_2021,3.185339,18.102384,1.016147,1.908326,1.325394,3.228025,25.185508,2.869013,6.714866,1.750739,0.814738,0.920917,13.254804,19.723800
D2_2020,1.914914,7.049384,0.795045,0.658233,1.789808,1.364800,11.365855,6.810309,21.223123,3.646133,1.217432,0.573340,10.806948,30.784678
D2_2021,2.260425,13.249129,0.712326,1.547870,1.379503,2.623595,14.582412,0.846074,29.131131,8.070595,0.890232,1.801655,5.772387,17.132665
D2_2022,2.838809,6.103901,0.678142,0.598402,1.106758,0.684252,10.453746,5.653759,19.951194,2.648666,2.171963,0.569689,18.821008,27.719710
D2_2023,1.601990,4.165016,0.405681,0.591231,0.783224,1.001421,6.598727,5.360586,21.966411,12.493033,1.000155,0.666650,10.306196,33.059679
D3_2022,4.741863,8.928896,0.520304,0.616935,0.618178,3.061851,6.424071,8.501051,4.060081,1.010903,9.719073,0.851398,10.785336,40.160061
D3_2023,2.347500,6.260507,0.752095,0.551558,0.721296,1.909146,13.589774,3.875740,5.141512,2.429789,16.821278,2.187251,7.730482,35.682074


### per site (over years)

In [9]:
df_site_perc = df_all[df_all["class"].isin(fig_class_order)]
df_site_perc

,class,site,year,site_year,class_num,count
0,Buxus,D1,2019,D1_2019,2,53.806745
1,Buxus,D1,2020,D1_2020,2,20.822339
2,Buxus,D1,2021,D1_2021,2,24.990120
3,Buxus,D2,2020,D2_2020,2,13.853471
4,Buxus,D2,2021,D2_2021,2,30.323475
...,...,...,...,...,...,...
499,VitisS,W4,2019,W4_2019,11,82.988265
500,VitisS,W4,2020,W4_2020,11,80.493538
501,VitisS,W4,2021,W4_2021,11,83.066276
502,VitisS,W4,2022,W4_2022,11,121.282832


In [10]:

df_site_perc= df_site_perc[["class", "site", "count"]].groupby(["class", "site"]).mean().reset_index()
df_site_perc.loc[:, "proportion"] = df_site_perc["count"] * 100 / df_site_perc.groupby(["site"])["count"].transform("sum")

df_site_perc

,class,site,count,proportion
0,Buxus,D1,33.206401,2.010150
1,Buxus,D2,19.023096,0.599816
2,Buxus,D3,33.128295,0.672984
3,Buxus,W1,54.320978,2.024713
4,Buxus,W2,10.695511,0.504405
...,...,...,...,...
93,VitisS,D3,85.225976,1.731322
94,VitisS,W1,12.088776,0.450587
95,VitisS,W2,21.080125,0.994147
96,VitisS,W3,12.973350,0.517567


In [11]:

df_site_perc = df_site_perc.pivot(index="site", columns="class", values="proportion").reset_index()
df_site_perc.columns.name = None
df_site_perc.set_index('site', inplace=True)

df_site_perc = df_site_perc[fig_class_order]
df_site_perc.reset_index()


df_site_perc


,QuercusDeciduous,QuercusIlex,Buxus,Phillyrea,Fraxinus,Olea,Cupressaceae,Pistacia,Poaceae,Plantago,VitisF,VitisS,Pinaceae,Other
site,,,,,,,,,,,,,,
D1,3.315166,13.195350,2.010150,1.529880,1.524171,2.365982,18.688970,10.976219,6.106940,1.520256,1.450416,0.962042,10.450439,25.904019
D2,2.037302,7.878176,0.599816,0.922443,1.166405,1.551729,10.466741,4.085424,23.989304,8.429744,1.155487,1.021266,10.033465,26.662700
D3,3.164701,7.171234,0.672984,0.573871,0.686101,2.302566,11.144105,5.454368,4.772417,1.945520,14.397280,1.731322,8.773110,37.210420
W1,3.174173,6.813410,2.024713,18.385130,12.296037,7.208284,3.690802,2.102663,1.684608,0.790910,12.746686,0.450587,7.148097,21.483900
W2,5.456543,41.706703,0.504405,0.520821,0.409102,0.889269,7.247409,2.579450,3.106562,1.080910,4.533795,0.994147,7.782291,23.188593
W3,2.685859,11.773600,30.038987,9.816750,8.907311,2.938533,4.068302,1.228093,1.716589,0.535995,1.252857,0.517567,6.016410,18.503146
W4,2.813056,7.582582,2.276709,8.493894,21.272516,3.413313,8.817093,3.697454,3.890950,1.039677,1.636614,2.809083,6.409400,25.847660


In [12]:
# saving
df_site_perc.to_csv(p_save_table / "percentages_persite_forAC_withOther.csv")



In [ ]:
# end